# Multi-Agent Research Pipeline

An automated research pipeline built with the **OpenAI Agents SDK**. Instead of one agent doing everything, two **specialized** agents each do one thing well:

1. **Researcher** — searches the web via `tavily_search` and returns up to 5 fact bullet points.
2. **Analyst** — takes those research notes (no tools) and extracts key **trends, risks, and insights** in 2 paragraphs.

A shared **Pydantic** model (`ResearchOutput`) lets data flow cleanly between them. Both agents use `gpt-5-mini`.

## 1. Environment setup

In [11]:
import os
from typing import Annotated

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient
from agents import Agent, Runner, function_tool

# Loads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv()

MODEL = "gpt-5-mini"  # used by both agents

## 2. Shared data model

Both agents return the **same** type, so the Researcher's output can be fed straight into the Analyst.

In [12]:
class ResearchOutput(BaseModel):
    """Structured output passed between agents in the pipeline."""

    summary: str = Field(
        description=(
            "For the Researcher: up to 5 bullet points of collected facts. "
            "For the Analyst: a 2-paragraph analysis of trends, risks, and insights."
        )
    )

## 3. The `tavily_search` tool

A function tool the Researcher calls to fetch real-time web results. It caps Tavily at 5 results and returns them as readable text for the model.

In [13]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@function_tool
def tavily_search(query: Annotated[str, "The web search query"]) -> str:
    """Search the web for real-time information about a topic."""
    response = tavily_client.search(query=query, max_results=5)
    results = response.get("results", [])
    if not results:
        return "No results found."
    return "\n\n".join(
        f"- {r['title']}\n  {r['content']}\n  Source: {r['url']}"
        for r in results
    )

## 4. Agent 1 — Researcher

Uses `tavily_search`, collects facts, and returns **at most 5 bullet points**.

In [14]:
researcher = Agent(
    name="Researcher",
    model=MODEL,
    instructions=(
        "You are a web researcher. Use the tavily_search tool to gather "
        "real-time, factual information on the user's topic. Collect the most "
        "important facts and return them as AT MOST 5 concise bullet points in "
        "the `summary` field. Stick to verifiable facts \u2014 do not analyze or "
        "speculate."
    ),
    tools=[tavily_search],
    output_type=ResearchOutput,
)

## 5. Agent 2 — Analyst

**No tools.** Receives the Researcher's notes and extracts trends, risks, and insights in **2 paragraphs**.

In [15]:
analyst = Agent(
    name="Analyst",
    model=MODEL,
    instructions=(
        "You are a research analyst. You receive research notes and extract the "
        "key trends, risks, and insights. Do NOT use any tools or invent facts "
        "beyond the notes. Write exactly TWO paragraphs in the `summary` field: "
        "paragraph 1 = trends and insights, paragraph 2 = risks and takeaways."
    ),
    output_type=ResearchOutput,
)

## 6. Test the Researcher

Run the Researcher on the test query and inspect its structured output.

In [16]:
query = "Why is Labubu so popular and what are the rarest Labubu collectibles?"

research_result = await Runner.run(researcher, query)
research_notes: ResearchOutput = research_result.final_output
print(research_notes.summary)

- Popularity: Labubu’s rise is driven by Pop Mart’s blind‑box sales model, Kasing Lung’s distinctive “ugly‑cute”/mischievous design, social‑media and celebrity exposure, and rapid global retail expansion (sold in 30+ countries; major sales growth in 2024–2025).
- Visibility boosters: High‑profile collaborations and releases (e.g., Coca‑Cola series, One Piece reimagining), Pop Mart events/Popland, and broad media coverage (BBC, NPR) increased demand.
- Secret/chase figures: Extremely low‑odds “secret” pulls (commonly 1/72 or 1/144) — e.g., Secret Have A Seat and Coca‑Cola secret variants — are among the rarest.
- Prototypes & uniques: Early artist resin prototypes and one‑of‑a‑kind PVC prototypes/giant sculptures (including reported life‑size/mint‑green and 131cm first‑generation uniques) are the most valuable and rare.
- Exclusives & regional runs: Popland exclusives, Japan‑only releases and limited collaboration runs (brand or event exclusives) rank among the rarest Labubu collectible

## 7. Full pipeline — hand the notes to the Analyst

The Researcher's `summary` flows into the Analyst as clean, typed data. This is the whole point of specialized agents: each does one job, and the Pydantic model is the contract between them.

In [17]:
analysis_result = await Runner.run(analyst, research_notes.summary)
analysis: ResearchOutput = analysis_result.final_output
print(analysis.summary)

Trends and insights: Labubu’s rise is driven by Pop Mart’s blind‑box sales model, Kasing Lung’s distinctive “ugly‑cute”/mischievous design, social‑media and celebrity exposure, and rapid global retail expansion (sold in 30+ countries with major sales growth in 2024–2025). High‑profile collaborations and releases (e.g., Coca‑Cola series, One Piece reimagining), Pop Mart events/Popland, and broad media coverage (BBC, NPR) have amplified demand. Rarity and value are concentrated in extremely low‑odds secret/chase figures (commonly 1/72 or 1/144), early artist resin prototypes and one‑of‑a‑kind PVC prototypes/giant sculptures (including reported life‑size/mint‑green and 131cm first‑generation uniques), and limited exclusives/region‑specific runs (Popland exclusives, Japan‑only releases, and collaboration/event exclusives).
Risks and takeaways: Acquisition and valuation risk is driven by extreme scarcity—secret pulls have very low odds and prototypes/uniques are singular—while regional excl

In [19]:
import os
from typing import Annotated

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient
from agents import Agent, Runner, function_tool, SQLiteSession

load_dotenv()
MODEL = "gpt-5-mini"

print("Kernel setup loaded. Defined:", [n for n in ["BaseModel","Field","Agent","Runner","function_tool","TavilyClient","MODEL"]])

Kernel setup loaded. Defined: ['BaseModel', 'Field', 'Agent', 'Runner', 'function_tool', 'TavilyClient', 'MODEL']


In [20]:
class ResearchOutput(BaseModel):
    """Structured output passed between agents in the pipeline."""

    summary: str = Field(
        description=(
            "For the Researcher: up to 5 bullet points of collected facts. "
            "For the Analyst: a 2-paragraph analysis of trends, risks, and insights."
        )
    )

print("ResearchOutput defined OK:", ResearchOutput.__name__, "| fields:", list(ResearchOutput.model_fields))

ResearchOutput defined OK: ResearchOutput | fields: ['summary']
